# Petunjuk menjalankan notebook yang dilengkapi

Sel awal tim tetap utuh sebagai alur sebelumnya. Untuk menjalankan delapan eksperimen dengan pembagian data dan metrik seragam, buka bagian **Tambahan eksperimen lengkap** setelah sel awal, lalu jalankan mulai dari sel setup bagian tersebut sampai akhir. Setiap langkah tambahan menjelaskan tujuan kode dan cara membaca hasilnya. Script `python src/execute_notebooks.py` menjalankan bagian tambahan secara otomatis.

In [2]:
import os, time, warnings, joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import ParameterGrid
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

In [3]:
SKIP_DIRS = {".git", ".ipynb_checkpoints", "__pycache__", "node_modules",
             ".venv", "venv", "env", ".idea", ".vscode"}

def find_file(filename, required_cols=None, prefer=("raw",), must_exist=True):
    """Cari `filename` di folder kerja, lalu naik ke folder induk sampai root repo (.git).
    Jika required_cols diberikan, hanya file yang header-nya memuat kolom tsb yang dianggap cocok."""
    start = Path.cwd().resolve()
    roots = []
    for p in [start, *start.parents]:
        roots.append(p)
        if (p / ".git").exists() or len(roots) >= 4:
            break

    for root in roots:
        matches = []
        for dirpath, dirnames, filenames in os.walk(root):
            dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS]
            if filename in filenames:
                path = Path(dirpath) / filename
                if required_cols:
                    try:
                        cols = pd.read_csv(path, nrows=0).columns
                    except Exception:
                        continue
                    if not set(required_cols).issubset(cols):
                        continue
                matches.append(path)
        if matches:
            matches.sort(key=lambda p: (not any(k in p.parts for k in prefer), len(p.parts)))
            if len(matches) > 1:
                print(f"Ditemukan {len(matches)} kandidat untuk '{filename}', dipakai yang pertama:")
                for m in matches:
                    print("   -", m)
            return matches[0]

    if must_exist:
        raise FileNotFoundError(
            f"'{filename}' tidak ditemukan. Dicari di: {[str(r) for r in roots]}. "
            "Pastikan notebook dijalankan dari dalam folder repo.")
    return None

DATA_PATH = find_file("train.csv", required_cols=["date", "food_waste_kg"], prefer=("raw",))
print("Dataset mentah dipakai:", DATA_PATH)

# ---------- Konfigurasi ----------
TARGET = "food_waste_kg"
SEED = 42
TEST_FRAC = 0.20        # 20% data terbaru -> test (sama dengan split sebelumnya: 728 train / 183 test)
VAL_FRAC = 0.15         # 15% data terbaru dari train -> validation (untuk tuning)
USE_LAG = False         # lag-1 target hanya bernilai kecil; aktifkan hanya jika ketersediaannya bisa dijustifikasi
CLIP_MEALS = False      # Robust: apakah meals_served ikut di-clip
SEEDS = [0, 1, 2, 3, 4] # seed untuk evaluasi akhir RF

# Folder output ditaruh di root repo (bukan di folder notebooks), dibuat otomatis
OUT_ROOT = Path.cwd().resolve()
for p in [OUT_ROOT, *OUT_ROOT.parents]:
    if (p / ".git").exists():
        OUT_ROOT = p
        break
RESULTS_DIR = OUT_ROOT / "results"
MODELS_DIR = OUT_ROOT / "models"
RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

Dataset mentah dipakai: D:\SDT\Project Sistem Cerdas\foodsafe\foodSaveProject\dataset\raw\train.csv


In [4]:
df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["date", "ID"]).reset_index(drop=True)

print("Shape:", df.shape)
print("Rentang tanggal:", df["date"].min().date(), "->", df["date"].max().date())
print("\nMissing value:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nDuplikat persis (tanpa ID):", df.drop(columns="ID").duplicated().sum())
print("\nNilai unik staff_experience :", df["staff_experience"].unique())
print("Nilai unik waste_category   :", df["waste_category"].unique())

if USE_LAG:
    df["lag1_waste"] = df[TARGET].shift(1)

Shape: (911, 12)
Rentang tanggal: 2022-01-01 -> 2024-09-26

Missing value:
 staff_experience    164
dtype: int64

Duplikat persis (tanpa ID): 0

Nilai unik staff_experience : ['Intermediate' 'intermediate' 'Beginner' nan 'EXPERT']
Nilai unik waste_category   : ['MEAT' 'GRAINS' 'Vegetables' 'MeAt' 'dairy']


In [5]:
n = len(df)
n_train = int(n * (1 - TEST_FRAC))
train_full = df.iloc[:n_train].copy()
test = df.iloc[n_train:].copy()

n_fit = int(len(train_full) * (1 - VAL_FRAC))
train_fit = train_full.iloc[:n_fit].copy()
val = train_full.iloc[n_fit:].copy()

for nm, d in [("train_fit", train_fit), ("val", val), ("train_full", train_full), ("test", test)]:
    print(f"{nm:10s} n={len(d):4d}  {d['date'].min().date()} -> {d['date'].max().date()}")

assert train_full["date"].max() <= test["date"].min(), "Train tidak boleh lebih baru dari test"
if train_full["date"].max() == test["date"].min():
    print("PERINGATAN: ada tanggal yang sama di batas train/test")

train_fit  n= 618  2022-01-01 -> 2023-10-23
val        n= 110  2023-10-24 -> 2024-02-28
train_full n= 728  2022-01-01 -> 2024-02-28
test       n= 183  2024-02-29 -> 2024-09-26


In [6]:
NUM_COLS = ["meals_served", "kitchen_staff", "temperature_C", "humidity_percent", "past_waste_kg"]
INT_COLS = ["day_of_week", "special_event"]
CAT_COLS = ["staff_experience", "waste_category"]


class StandardPrep:
    """Standard: konversi tipe, imputasi sederhana (median / modus), one-hot kategori APA ADANYA
    (variasi penulisan tidak dinormalisasi), tanpa clipping, tanpa hapus duplikat.
    Semua statistik dihitung dari data yang di-fit saja (tanpa kebocoran)."""
    name = "standard"

    def __init__(self):
        self.num_cols = NUM_COLS + INT_COLS + (["lag1_waste"] if USE_LAG else [])

    # ---- hook yang dioverride Robust ----
    def _clean_cat(self, d, fit=False):
        for c in CAT_COLS:
            if fit:
                self.modes_[c] = d[c].mode(dropna=True)[0]
            d[c] = d[c].fillna(self.modes_[c])
        return d

    def _fit_extra(self, d): pass
    def _after_numeric(self, d): return d
    def _prepare_train_rows(self, df): return df

    # ---- API ----
    def fit_transform(self, df):
        df = self._prepare_train_rows(df)
        d = df.copy()
        self.modes_ = {}
        for c in self.num_cols:
            d[c] = pd.to_numeric(d[c], errors="coerce")
        self.medians_ = d[self.num_cols].median()
        d[self.num_cols] = d[self.num_cols].fillna(self.medians_)
        self._fit_extra(d)
        d = self._after_numeric(d)
        d = self._clean_cat(d, fit=True)
        self.cat_levels_ = {c: sorted(d[c].dropna().unique()) for c in CAT_COLS}
        return self.transform(df, _fitted_frame=d)

    def transform(self, df, _fitted_frame=None):
        if _fitted_frame is None:
            d = df.copy()
            for c in self.num_cols:
                d[c] = pd.to_numeric(d[c], errors="coerce")
            d[self.num_cols] = d[self.num_cols].fillna(self.medians_)
            d = self._after_numeric(d)
            d = self._clean_cat(d)
        else:
            d = _fitted_frame
        # fitur turunan (sama untuk Standard dan Robust agar perbandingan adil)
        d["is_weekend"] = (d["day_of_week"] >= 5).astype(int)
        d["month"] = pd.to_datetime(d["date"]).dt.month
        d["meals_per_staff"] = d["meals_served"] / d["kitchen_staff"].replace(0, np.nan)
        d["meals_per_staff"] = d["meals_per_staff"].fillna(d["meals_per_staff"].median())
        cols = self.num_cols + ["is_weekend", "month", "meals_per_staff"]
        X = d[cols].copy()
        for c, levels in self.cat_levels_.items():
            for lv in levels:
                X[f"{c}={lv}"] = (d[c] == lv).astype(int)   # kategori tak dikenal -> semua 0
        y = d[TARGET]
        return X, y


class RobustPrep(StandardPrep):
    """Robust: normalisasi kategori (strip + lowercase), missing kategori -> 'unknown',
    hapus duplikat persis HANYA pada data training, winsorizing outlier fitur (batas dari train,
    diterapkan KONSISTEN ke train & test). Target tidak pernah diubah."""
    name = "robust"
    IQR_K = 3.0

    def _clean_cat(self, d, fit=False):
        for c in CAT_COLS:
            d[c] = d[c].astype("string").str.strip().str.lower().fillna("unknown").astype(str)
        return d

    def _prepare_train_rows(self, df):
        key = [c for c in df.columns if c != "ID"]
        out = df.drop_duplicates(subset=key)
        self.n_dup_removed_ = len(df) - len(out)
        return out

    def _fit_extra(self, d):
        clip_cols = ["kitchen_staff", "temperature_C", "humidity_percent", "past_waste_kg"]
        if CLIP_MEALS:
            clip_cols.append("meals_served")
        self.clip_bounds_ = {}
        for c in clip_cols:
            q1, q3 = d[c].quantile(0.25), d[c].quantile(0.75)
            iqr = q3 - q1
            self.clip_bounds_[c] = (q1 - self.IQR_K * iqr, q3 + self.IQR_K * iqr)

    def _after_numeric(self, d):
        for c, (lo, hi) in self.clip_bounds_.items():
            d[c] = d[c].clip(lo, hi)
        return d

In [7]:
PREPS = {"standard": StandardPrep, "robust": RobustPrep}
TUNE, FINAL, PREP_FINAL = {}, {}, {}

for name, P in PREPS.items():
    # (a) untuk tuning: fit di train_fit, evaluasi di val
    p_tune = P()
    Xtr, ytr = p_tune.fit_transform(train_fit)
    Xva, yva = p_tune.transform(val)
    TUNE[name] = (Xtr, ytr, Xva, yva)
    # (b) untuk evaluasi akhir: fit di train_full, evaluasi di test (dipakai sekali)
    p_fin = P()
    Xtf, ytf = p_fin.fit_transform(train_full)
    Xte, yte = p_fin.transform(test)
    FINAL[name] = (Xtf, ytf, Xte, yte)
    PREP_FINAL[name] = p_fin

In [8]:
rows = []
for name in PREPS:
    Xtf, ytf, Xte, yte = FINAL[name]
    Xtr, ytr, Xva, yva = TUNE[name]
    assert list(Xtf.columns) == list(Xte.columns) and list(Xtr.columns) == list(Xva.columns)
    assert Xtf.isna().sum().sum() == 0 and Xte.isna().sum().sum() == 0, "Masih ada NaN"
    assert "ID" not in Xtf.columns and "date" not in Xtf.columns
    # cek bug lama: special_event harus punya nilai 0 DAN 1 di train maupun test
    assert Xtf["special_event"].nunique() == 2 and Xte["special_event"].nunique() == 2, "special_event hilang!"
    rows.append({
        "preprocessing": name, "n_train": len(Xtf), "n_test": len(Xte), "n_fitur": Xtf.shape[1],
        "event_train": int(Xtf["special_event"].sum()), "event_test": int(Xte["special_event"].sum()),
        "dup_dihapus": getattr(PREP_FINAL[name], "n_dup_removed_", 0),
        "meals_max_train": Xtf["meals_served"].max(), "meals_max_test": Xte["meals_served"].max(),
    })
print(pd.DataFrame(rows).to_string(index=False))

# test set harus IDENTIK antara Standard dan Robust (perbandingan adil)
assert FINAL["standard"][3].index.equals(FINAL["robust"][3].index)
assert np.allclose(FINAL["standard"][3].values, FINAL["robust"][3].values)
print("\nOK: baris & target test identik di kedua preprocessing")

print("\nBatas clipping Robust (dari train):")
for c, (lo, hi) in PREP_FINAL["robust"].clip_bounds_.items():
    n_tr = ((train_full[c] < lo) | (train_full[c] > hi)).sum()
    n_te = ((test[c] < lo) | (test[c] > hi)).sum()
    print(f"  {c:18s} [{lo:8.2f}, {hi:8.2f}]  di luar batas: train={n_tr}, test={n_te}")

print("\nKolom Standard:", list(FINAL["standard"][0].columns))
print("Kolom Robust  :", list(FINAL["robust"][0].columns))

preprocessing  n_train  n_test  n_fitur  event_train  event_test  dup_dihapus  meals_max_train  meals_max_test
     standard      728     183       19           63          15            0             4730            4100
       robust      728     183       18           63          15            0             4730            4100

OK: baris & target test identik di kedua preprocessing

Batas clipping Robust (dari train):
  kitchen_staff      [  -13.00,    36.00]  di luar batas: train=0, test=0
  temperature_C      [  -23.38,    67.85]  di luar batas: train=0, test=0
  humidity_percent   [  -43.22,   165.28]  di luar batas: train=0, test=0
  past_waste_kg      [  -50.62,   104.89]  di luar batas: train=0, test=0

Kolom Standard: ['meals_served', 'kitchen_staff', 'temperature_C', 'humidity_percent', 'past_waste_kg', 'day_of_week', 'special_event', 'is_weekend', 'month', 'meals_per_staff', 'staff_experience=Beginner', 'staff_experience=EXPERT', 'staff_experience=Intermediate', 'staff_exp

In [9]:
old_te_path = find_file("others_test_standard_target.csv", must_exist=False, prefer=("processed",))
old_tr_path = find_file("others_train_standard_target.csv", must_exist=False, prefer=("processed",))

if old_te_path and old_tr_path:
    old_te = pd.read_csv(old_te_path)["food_waste_kg"].values
    old_tr = pd.read_csv(old_tr_path)["food_waste_kg"].values
    print("test sama dengan split lamamu :", np.allclose(np.sort(old_te), np.sort(test[TARGET].values)))
    print("train sama dengan split lamamu:", np.allclose(np.sort(old_tr), np.sort(train_full[TARGET].values)))
else:
    print("File preprocessing lama tidak ditemukan di repo, lewati cek ini.")

test sama dengan split lamamu : True
train sama dengan split lamamu: True


In [10]:
def metrics(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
            "R2": r2_score(y_true, y_pred)}

grid = list(ParameterGrid({
    "n_estimators": [300, 500],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", 0.5, 1.0],
}))

best_params, tune_rows = {}, []
for name, (Xtr, ytr, Xva, yva) in TUNE.items():
    best = (np.inf, None)
    for params in grid:
        m = RandomForestRegressor(**params, random_state=SEED, n_jobs=-1).fit(Xtr, ytr)
        s = metrics(yva, m.predict(Xva))
        tune_rows.append({"preprocessing": name, "params": str(params), **{f"val_{k}": v for k, v in s.items()}})
        if s["MAE"] < best[0]:
            best = (s["MAE"], params)
    best_params[name] = best[1]
    print(f"[{name}] best val MAE = {best[0]:.3f}  params = {best[1]}")

tune_df = pd.DataFrame(tune_rows)
tune_df.to_csv(RESULTS_DIR / "rf_tuning_validation.csv", index=False)
tune_df.sort_values("val_MAE").groupby("preprocessing").head(5)

[standard] best val MAE = 5.390  params = {'max_depth': 10, 'max_features': 1.0, 'min_samples_leaf': 1, 'n_estimators': 500}
[robust] best val MAE = 5.408  params = {'max_depth': 10, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 300}


,preprocessing,params,val_MAE,val_RMSE,val_R2
31,standard,"{'max_depth': 10, 'max_features': 1.0, 'min_sa...",5.390216,8.339676,0.947020
78,robust,"{'max_depth': 10, 'max_features': 0.5, 'min_sa...",5.407705,8.920527,0.939383
98,robust,"{'max_depth': 20, 'max_features': 0.5, 'min_sa...",5.408133,8.932071,0.939226
62,robust,"{'max_depth': None, 'max_features': 0.5, 'min_...",5.408133,8.932071,0.939226
85,robust,"{'max_depth': 10, 'max_features': 1.0, 'min_sa...",5.408519,8.420849,0.945983
67,robust,"{'max_depth': None, 'max_features': 1.0, 'min_...",5.409789,8.328556,0.947161
49,standard,"{'max_depth': 20, 'max_features': 1.0, 'min_sa...",5.429940,8.496033,0.945015
13,standard,"{'max_depth': None, 'max_features': 1.0, 'min_...",5.435249,8.490151,0.945091
30,standard,"{'max_depth': 10, 'max_features': 1.0, 'min_sa...",5.438808,8.350418,0.946883
48,standard,"{'max_depth': 20, 'max_features': 1.0, 'min_sa...",5.450865,8.385267,0.946439


In [11]:
final_rows, final_models = [], {}

for name, (Xtf, ytf, Xte, yte) in FINAL.items():
    per_seed = []
    for s in SEEDS:
        m = RandomForestRegressor(**best_params[name], random_state=s, n_jobs=-1)
        t0 = time.perf_counter(); m.fit(Xtf, ytf); t_train = time.perf_counter() - t0
        t0 = time.perf_counter(); pred = m.predict(Xte); t_pred = time.perf_counter() - t0
        per_seed.append({**metrics(yte, pred), "train_time_s": t_train, "pred_time_s": t_pred})
        if s == SEEDS[0]:
            final_models[name] = (m, pred)
    ps = pd.DataFrame(per_seed)
    final_rows.append({"model": "RandomForest", "preprocessing": name,
                       **{f"{c}_mean": ps[c].mean() for c in ps.columns},
                       **{f"{c}_std": ps[c].std() for c in ["MAE", "RMSE", "R2"]}})

# pembanding: tebak rata-rata & regresi linear (Ridge)
for name, (Xtf, ytf, Xte, yte) in FINAL.items():
    for label, mdl in [("Baseline-Mean", DummyRegressor(strategy="mean")),
                       ("Ridge", make_pipeline(StandardScaler(), Ridge(alpha=1.0)))]:
        t0 = time.perf_counter(); mdl.fit(Xtf, ytf); t_train = time.perf_counter() - t0
        t0 = time.perf_counter(); pred = mdl.predict(Xte); t_pred = time.perf_counter() - t0
        final_rows.append({"model": label, "preprocessing": name,
                           **{f"{k}_mean": v for k, v in metrics(yte, pred).items()},
                           "train_time_s_mean": t_train, "pred_time_s_mean": t_pred})

results = pd.DataFrame(final_rows)
results.to_csv(RESULTS_DIR / "rf_experiment_results.csv", index=False)
print(results.round(3).to_string(index=False))

for name, (m, _) in final_models.items():
    joblib.dump({"prep": PREP_FINAL[name], "model": m, "params": best_params[name]},
                MODELS_DIR / f"rf_{name}.joblib")

        model preprocessing  MAE_mean  RMSE_mean  R2_mean  train_time_s_mean  pred_time_s_mean  MAE_std  RMSE_std  R2_std
 RandomForest      standard     5.285      8.164    0.898              4.768             0.529    0.018     0.059   0.001
 RandomForest        robust     4.858      6.655    0.932              2.421             0.365    0.054     0.167   0.003
Baseline-Mean      standard    13.653     25.556   -0.004              0.015             0.001      NaN       NaN     NaN
        Ridge      standard     4.909      7.727    0.908              0.413             0.021      NaN       NaN     NaN
Baseline-Mean        robust    13.653     25.556   -0.004              0.005             0.000      NaN       NaN     NaN
        Ridge        robust     4.904      7.716    0.908              0.026             0.008      NaN       NaN     NaN


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (name, (m, _)) in zip(axes, final_models.items()):
    imp = pd.Series(m.feature_importances_, index=FINAL[name][0].columns).sort_values().tail(10)
    imp.plot.barh(ax=ax, title=f"Impurity importance - RF ({name})")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "rf_feature_importance.png", dpi=120); plt.show()

for name, (m, _) in final_models.items():
    Xte, yte = FINAL[name][2], FINAL[name][3]
    pi = permutation_importance(m, Xte, yte, scoring="neg_mean_absolute_error",
                                n_repeats=10, random_state=SEED, n_jobs=-1)
    top = pd.Series(pi.importances_mean, index=Xte.columns).sort_values(ascending=False).head(8)
    print(f"\nPermutation importance (kenaikan MAE jika fitur diacak) - {name}")
    print(top.round(3).to_string())


Permutation importance (kenaikan MAE jika fitur diacak) - standard
meals_served           7.253
past_waste_kg          5.345
meals_per_staff        1.296
special_event          0.626
humidity_percent       0.143
temperature_C          0.132
day_of_week            0.031
waste_category=MEAT    0.009

Permutation importance (kenaikan MAE jika fitur diacak) - robust
meals_served            6.588
past_waste_kg           5.274
meals_per_staff         2.189
special_event           0.713
temperature_C           0.152
humidity_percent        0.123
waste_category=dairy    0.017
day_of_week             0.009


In [13]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for j, (name, (m, pred)) in enumerate(final_models.items()):
    yte = FINAL[name][3]
    axes[0, j].scatter(yte, pred, s=14, alpha=0.6)
    lim = [yte.min(), yte.max()]
    axes[0, j].plot(lim, lim, "r--"); axes[0, j].set_title(f"Aktual vs Prediksi ({name})")
    axes[0, j].set_xlabel("Aktual (kg)"); axes[0, j].set_ylabel("Prediksi (kg)")
    axes[1, j].scatter(pred, yte - pred, s=14, alpha=0.6); axes[1, j].axhline(0, color="r", ls="--")
    axes[1, j].set_title(f"Residual ({name})")
    axes[1, j].set_xlabel("Prediksi (kg)"); axes[1, j].set_ylabel("Residual (kg)")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "rf_actual_vs_pred.png", dpi=120); plt.show()

# error pada baris dengan meals_served ekstrem (>1000) vs normal
for name, (m, pred) in final_models.items():
    Xte, yte = FINAL[name][2], FINAL[name][3]
    ext = Xte["meals_served"] > 1000
    print(f"[{name}] MAE meals<=1000: {mean_absolute_error(yte[~ext], pred[~ext.values]):.2f} (n={int((~ext).sum())})"
          f" | meals>1000: {mean_absolute_error(yte[ext], pred[ext.values]):.2f} (n={int(ext.sum())})")

[standard] MAE meals<=1000: 4.41 (n=178) | meals>1000: 35.85 (n=5)
[robust] MAE meals<=1000: 4.34 (n=178) | meals>1000: 25.86 (n=5)


## Tambahan eksperimen lengkap

Kode awal milik tim di atas tetap dipertahankan tanpa penghapusan. Bagian di bawah menyediakan alur tambahan yang dapat dijalankan mulai dari sel setup hingga akhir, tanpa menjalankan ulang sel lama. Hasil alur lama tidak dicampur karena pembagian data dan asumsi fiturnya berbeda. Script eksekusi otomatis melewati sel referensi lama bertag `foodsave-legacy`.

# Model RandomForest untuk prediksi limbah makanan

**Tujuan notebook:** menyelesaikan dua eksperimen RandomForest dengan Standard dan Robust secara terbuka, langkah demi langkah.

Random Forest menggabungkan prediksi banyak pohon keputusan. Setiap pohon mempelajari aturan seperti pembagian jumlah porsi. Kandidat membandingkan kedalaman tak dibatasi/10 dan minimal isi daun 1/3. Jumlah pohon bertambah 20, 60, 120, 200 untuk memperlihatkan kestabilan error. Model ini tidak memakai gradient descent.

**Pertanyaan yang dijawab:** Seberapa besar kesalahan dalam kg? Apakah Robust membantu? Apakah training terlalu bagus dibanding validasi? Fitur apa yang membantu model? Seberapa cepat prediksinya? Semua angka kesimpulan dihasilkan dari kode, bukan angka ilustrasi.

## Langkah 1 Menyiapkan alat dan konfigurasi
`numpy` mengolah angka, `pandas` mengolah tabel, `matplotlib` membuat grafik, dan `joblib` menyimpan model. Seed 42 membantu pengulangan. Fungsi bersama dipakai untuk menjaga definisi metrik, data, dan preprocessing tetap sama di empat notebook. Bagian pelatihan model ditulis langsung di notebook ini.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / 'src/experiment.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'src/experiment.py').exists(), 'Buka notebook dari folder proyek FoodSave.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown, Code
from src.experiment import OUT, ART, load_data, prepare, run_model, evaluate
pd.set_option('display.max_columns', 40)
pd.set_option('display.precision', 4)
get_ipython().run_line_magic('matplotlib', 'inline')

import time
import hashlib
import inspect
import joblib
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from src.experiment import NUM, CAT, TARGET, SEED, WINDOW, FoodPreprocessor, sequence, metrics, predict, plot_curve, save_json
MODEL = 'RandomForest'
CONFIGS = [{'max_depth':None,'min_samples_leaf':1,'max_features':1.0}, {'max_depth':10,'min_samples_leaf':3,'max_features':.8}]


## Langkah 2 Membaca data dan memastikan urutan waktu
Training memakai tanggal awal, validasi tanggal berikutnya, dan test tanggal terbaru. Tidak ada tanggal yang masuk dua partisi. Duplikat persis tanpa ID dibuang untuk semua model agar tidak muncul salinan observasi yang sama pada train/test. Dataset test Kaggle tanpa label tidak digunakan sebagai holdout penelitian.

In [2]:
df = load_data()
display(df.head(8))
display(df.groupby('split').agg(jumlah=('date','size'), tanggal_awal=('date','min'), tanggal_akhir=('date','max')))
display(json.loads((OUT/'data_audit.json').read_text(encoding='utf-8')))
assert df.loc[df.split=='train','date'].max() < df.loc[df.split=='validation','date'].min()
assert df.loc[df.split=='validation','date'].max() < df.loc[df.split=='test','date'].min()


,ID,date,meals_served,kitchen_staff,temperature_C,humidity_percent,day_of_week,special_event,past_waste_kg,staff_experience,waste_category,food_waste_kg,_row,split
0,397,2022-01-01,202,9,31.9768,66.8266,5,0,45.4455,Intermediate,MEAT,39.9011,341,train
1,427,2022-01-01,202,9,32.3188,66.8266,5,0,45.4455,Intermediate,MEAT,39.9011,364,train
2,201,2022-01-02,448,16,10.0925,56.1303,6,0,29.8900,intermediate,GRAINS,53.2034,171,train
3,983,2022-01-02,448,16,10.1286,56.1303,6,0,29.8900,intermediate,GRAINS,53.2034,849,train
4,162,2022-01-03,370,14,30.4987,64.6350,0,0,41.5516,Beginner,Vegetables,61.3754,141,train
5,764,2022-01-03,370,14,30.7597,64.6350,0,0,41.5516,Beginner,Vegetables,61.3754,656,train
6,50,2022-01-04,206,11,15.7402,74.2841,1,0,22.2819,Beginner,MeAt,32.4411,47,train
7,112,2022-01-04,206,11,15.7156,74.2841,1,0,22.2819,Beginner,MeAt,32.4411,99,train


,jumlah,tanggal_awal,tanggal_akhir
split,,,
test,174,2024-03-11,2024-09-26
train,564,2022-01-01,2023-08-23
validation,173,2023-08-24,2024-03-10


{'source_sha256': 'c2bedb198bf40cc6793921cdfe0573589923996b3347b3361561b114214b9adf',
 'raw_rows': 911,
 'invalid_date_or_target_removed': 0,
 'exact_duplicates_removed_for_all_models': 0,
 'usable_rows': 911,
 'missing_before': {'ID': 0,
  'date': 0,
  'meals_served': 0,
  'kitchen_staff': 0,
  'temperature_C': 0,
  'humidity_percent': 0,
  'day_of_week': 0,
  'special_event': 0,
  'past_waste_kg': 0,
  'staff_experience': 164,
  'waste_category': 0,
  'food_waste_kg': 0},
 'excluded_features': ['ID', 'waste_category', 'past_waste_kg'],
 'sequence_warning': '7 observasi, bukan 7 hari. Tidak ada ID restoran; urutan temporal tidak membuktikan satu seri restoran.',
 'split': {'test': {'n': 174, 'start': '2024-03-11', 'end': '2024-09-26'},
  'train': {'n': 564, 'start': '2022-01-01', 'end': '2023-08-23'},
  'validation': {'n': 173, 'start': '2023-08-24', 'end': '2024-03-10'}}}

## Langkah 3 Memahami input dan mencegah kebocoran informasi
Hanya informasi yang diasumsikan tersedia ketika merencanakan operasional yang digunakan. ID, kategori limbah, dan limbah sebelumnya dikeluarkan; asal/waktu ketersediaannya belum terverifikasi. Hari dihitung dari tanggal. Suhu/kelembapan harus berupa perkiraan jika nilai aktual belum diketahui. Target food_waste_kg tidak pernah menjadi input.

Standard memakai konversi sederhana, imputasi median/modus training, dan kategori apa adanya. Robust menambah pembersihan simbol/spasi, normalisasi kategori, batas IQR training, dan RobustScaler untuk LSTM. Model pohon tidak memerlukan scaling. Target aktual tidak dipotong. Rasio dihitung setelah pembersihan agar tidak membagi dengan nilai nol.

In [3]:
display(Code(inspect.getsource(FoodPreprocessor), language='python'))

class FoodPreprocessor:
    """Standard: konversi sederhana. Robust: normalisasi dan batas IQR training."""
    def __init__(self, method='standard'):
        self.method = method

    def clean(self, df):
        d = df[NUM + CAT].copy()
        for c in NUM:
            s = d[c]
            if self.method == 'robust':
                s = s.astype('string').str.strip().str.replace(',', '', regex=False).str.replace(r'[^0-9.eE+\-]', '', regex=True)
            # Float mendukung median/batas IQR pecahan, termasuk pada pandas 3.
            d[c] = pd.to_numeric(s, errors='coerce').astype('float64').replace([np.inf, -np.inf], np.nan)
        for c in CAT:
            s = d[c].astype('string')
            if self.method == 'robust':
                s = s.str.strip().str.lower().replace({'nan': pd.NA, 'none': pd.NA, '': pd.NA})
            d[c] = s
        date = pd.to_datetime(df['date'])
        d['is_weekend'] = (date.dt.dayofweek >= 5).astype(float)
        d['month_sin'] = np.sin(2 * np.pi * date.dt.month / 12)
        d['month_cos'] = np.cos(2 * np.pi * date.dt.month / 12)
        return d

    def fit(self, df):
        d = self.clean(df)
        self.numerics = d.select_dtypes(include='number').columns.tolist()
        self.medians = d[self.numerics].median().fillna(0)
        self.modes = {c: (d[c].dropna().mode().iloc[0] if d[c].notna().any() else 'unknown') for c in CAT}
        self.bounds = {}
        if self.method == 'robust':
            for c in ['meals_served', 'kitchen_staff', 'temperature_C', 'humidity_percent']:
                q1, q3 = d[c].quantile([.25, .75]); iqr = q3 - q1
                self.bounds[c] = (float(q1 - 1.5 * iqr), float(q3 + 1.5 * iqr))
        native = self.native(df)
        self.columns = pd.get_dummies(native, columns=CAT, dtype=float).columns.tolist()
        self.scaler = (RobustScaler() if self.method == 'robust' else StandardScaler())
        self.scaler.fit(self.encoded(df))
        return self

    def native(self, df):
        d = self.clean(df)
        d[self.numerics] = d[self.numerics].fillna(self.medians)
        for c, bounds in self.bounds.items():
            d[c] = d[c].clip(*bounds)
        for c in CAT:
            d[c] = d[c].fillna(self.modes[c]).astype(str)
        # Rasio dihitung setelah imputasi dan penanganan outlier.
        d['staff_per_portion'] = d.kitchen_staff / d.meals_served.clip(lower=1)
        d['portion_per_staff'] = d.meals_served / d.kitchen_staff.clip(lower=1)
        d['special_event_portion'] = d.special_event * d.meals_served
        return d

    def encoded(self, df):
        return pd.get_dummies(self.native(df), columns=CAT, dtype=float).reindex(columns=self.columns, fill_value=0).astype(float)

    def transform(self, df, scaled=False):
        x = self.encoded(df)
        return self.scaler.transform(x).astype('float32') if scaled else x

## Langkah 4 Menyiapkan fitur dan baris evaluasi yang setara
Statistik preprocessing dipelajari dari training saja, lalu dipakai pada validasi/test. Tujuh observasi pertama diperlukan untuk jendela LSTM. Enam baris awal tidak dijadikan target pada seluruh model agar baris pembanding identik. Kategori CatBoost tetap berupa teks; RF/XGBoost memakai one-hot. LSTM memakai urutan fitur terskala. Tidak ada label test yang digunakan untuk konteks.

In [4]:
eligible = np.arange(len(df)) >= WINDOW-1
idx = {s: np.flatnonzero((df.split==s).to_numpy() & eligible) for s in ['train','validation','test']}
data = {}
for method in ['standard','robust']:
    prep = FoodPreprocessor(method).fit(df[df.split=='train'])
    x = prep.transform(df)
    scaled = prep.transform(df, scaled=True)
    native = prep.native(df)
    target_scaler = StandardScaler().fit(df.loc[idx['train'],TARGET].to_numpy().reshape(-1,1))
    y = {s:df.loc[ix,TARGET].to_numpy() for s,ix in idx.items()}
    if MODEL=='LSTM':
        features = {s:sequence(scaled,ix) for s,ix in idx.items()}
    elif MODEL=='CatBoost':
        features = {s:native.iloc[ix] for s,ix in idx.items()}
    else:
        features = {s:x.iloc[ix] for s,ix in idx.items()}
    assert np.isfinite(x.to_numpy()).all() and np.isfinite(scaled).all()
    assert not set(['ID',TARGET,'waste_category','past_waste_kg']).intersection(x.columns)
    data[method] = {'prep':prep,'ys':target_scaler,'y':y,'features':features,'names':native.columns.tolist() if MODEL=='CatBoost' else x.columns.tolist()}
    print(method, '| jumlah fitur:', x.shape[1], '| bentuk input:', {s:v.shape for s,v in features.items()})
    display(native.head(4))


standard | jumlah fitur: 15 | bentuk input: {'train': (558, 15), 'validation': (173, 15), 'test': (174, 15)}


,meals_served,kitchen_staff,temperature_C,humidity_percent,special_event,staff_experience,is_weekend,month_sin,month_cos,staff_per_portion,portion_per_staff,special_event_portion
0,202.0,9.0,31.9768,66.8266,0.0,Intermediate,1.0,0.5,0.866,0.0446,22.4444,0.0
1,202.0,9.0,32.3188,66.8266,0.0,Intermediate,1.0,0.5,0.866,0.0446,22.4444,0.0
2,448.0,16.0,10.0925,56.1303,0.0,intermediate,1.0,0.5,0.866,0.0357,28.0000,0.0
3,448.0,16.0,10.1286,56.1303,0.0,intermediate,1.0,0.5,0.866,0.0357,28.0000,0.0


robust | jumlah fitur: 14 | bentuk input: {'train': (558, 14), 'validation': (173, 14), 'test': (174, 14)}


,meals_served,kitchen_staff,temperature_C,humidity_percent,special_event,staff_experience,is_weekend,month_sin,month_cos,staff_per_portion,portion_per_staff,special_event_portion
0,202.0,9.0,31.9768,66.8266,0.0,intermediate,1.0,0.5,0.866,0.0446,22.4444,0.0
1,202.0,9.0,32.3188,66.8266,0.0,intermediate,1.0,0.5,0.866,0.0446,22.4444,0.0
2,448.0,16.0,10.0925,56.1303,0.0,intermediate,1.0,0.5,0.866,0.0357,28.0000,0.0
3,448.0,16.0,10.1286,56.1303,0.0,intermediate,1.0,0.5,0.866,0.0357,28.0000,0.0


## Langkah 5 Menulis cara pelatihan model
Fungsi berikut berisi kode model yang benar-benar dilatih, bukan placeholder. Training memperbarui parameter dari data latihan. Validasi memantau kemampuan di periode berikutnya. Early stopping menghentikan jaringan/boosting ketika validasi tidak membaik, lalu memakai keadaan terbaik. RF melihat kestabilan error ketika jumlah pohon bertambah.

Dua konfigurasi diuji per metode, bukan pencarian optimum global. Learning rate adalah besar langkah belajar; kedalaman pohon mengatur kerumitan aturan; regularisasi/minimum daun membatasi model agar tidak terlalu menyesuaikan training.

In [5]:
def latih_kandidat(config, xtr, ytr, xva, yva, target_scaler):
    start = time.perf_counter()
    model = RandomForestRegressor(random_state=SEED, n_jobs=2, warm_start=True, **config)
    rows = []
    for n in [20, 60, 120, 200]:
        model.set_params(n_estimators=n).fit(xtr, ytr)
        rows.append({'step': n, 'train_RMSE': metrics(ytr, model.predict(xtr))['RMSE'], 'validation_RMSE': metrics(yva, model.predict(xva))['RMSE']})
    curve = pd.DataFrame(rows)
    return model, curve, time.perf_counter() - start

## Langkah 6 Mencoba kandidat dan menghitung skor validasi
Bagian ini membutuhkan waktu komputasi. Semua kandidat dilatih hanya pada training. Masing-masing dibandingkan menggunakan MAE validasi dalam kg. Data test belum digunakan untuk memilih konfigurasi. Kurva training/validasi dan waktu setiap kandidat dicatat agar prosesnya dapat diaudit.

In [6]:
candidates = {}
tuning_rows = []
for method, d in data.items():
    candidates[method] = []
    for config in CONFIGS:
        model, curve, duration = latih_kandidat(config, d['features']['train'], d['y']['train'], d['features']['validation'], d['y']['validation'], d['ys'])
        prediction = predict(model, d['features']['validation'], MODEL, d['ys'])
        score = metrics(d['y']['validation'], prediction)
        tuning_rows.append({'preprocessing':method, 'config':json.dumps(config), **score, 'train_time_s':duration})
        candidates[method].append({'model':model,'curve':curve,'duration':duration,'config':config,'score':score})
        print(method, config, '| MAE validasi:', round(score['MAE'],4), 'kg', flush=True)
tuning = pd.DataFrame(tuning_rows)
display(tuning.sort_values(['preprocessing','MAE']))


standard {'max_depth': None, 'min_samples_leaf': 1, 'max_features': 1.0} | MAE validasi: 9.9618 kg


standard {'max_depth': 10, 'min_samples_leaf': 3, 'max_features': 0.8} | MAE validasi: 9.8086 kg


robust {'max_depth': None, 'min_samples_leaf': 1, 'max_features': 1.0} | MAE validasi: 9.2478 kg


robust {'max_depth': 10, 'min_samples_leaf': 3, 'max_features': 0.8} | MAE validasi: 9.7018 kg


,preprocessing,config,MAE,RMSE,R2,MedAE,Bias,WAPE_percent,train_time_s
2,robust,"{""max_depth"": null, ""min_samples_leaf"": 1, ""ma...",9.2478,12.6380,0.8670,7.8892,-0.7663,19.4660,0.7363
3,robust,"{""max_depth"": 10, ""min_samples_leaf"": 3, ""max_...",9.7018,13.1971,0.8550,8.1969,-0.5418,20.4217,0.5810
1,standard,"{""max_depth"": 10, ""min_samples_leaf"": 3, ""max_...",9.8086,13.7094,0.8435,7.5323,-0.6138,20.6464,0.5859
0,standard,"{""max_depth"": null, ""min_samples_leaf"": 1, ""ma...",9.9618,14.0523,0.8356,7.8437,-0.6193,20.9688,0.5831


## Langkah 7 Mengunci konfigurasi dan membaca kurva belajar
MAE validasi terendah memilih satu kandidat pada Standard dan satu pada Robust. Kurva yang turun berarti error mengecil; kurva tidak harus selalu turun. Error training yang jauh lebih rendah daripada validasi bisa menandakan overfitting atau perubahan distribusi waktu. Sumbu RF adalah jumlah pohon, sumbu boosting adalah tahap, dan sumbu LSTM adalah epoch. RF tidak memakai gradient descent.

In [7]:
selected = {method:min(items,key=lambda item:item['score']['MAE']) for method,items in candidates.items()}
for method,item in selected.items():
    label=f'{MODEL}_{method}'
    item['curve'].to_csv(OUT/f'{label}_curve.csv',index=False)
    tuning[tuning.preprocessing==method].drop(columns='preprocessing').to_csv(OUT/f'{label}_tuning.csv',index=False)
    plot_curve(item['curve'],label)
    print('Terpilih:',method,item['config'])
    display(Image(filename=str(OUT/f'{label}_curve.png')))


Terpilih: standard {'max_depth': 10, 'min_samples_leaf': 3, 'max_features': 0.8}


Terpilih: robust {'max_depth': None, 'min_samples_leaf': 1, 'max_features': 1.0}


## Langkah 8 Mengukur seluruh metrik training validasi dan test
MAE: kesalahan absolut rata-rata dalam kg. RMSE: lebih sensitif terhadap error besar. R²: kemampuan menjelaskan variasi; dapat negatif. MedAE: median error absolut. Bias: prediksi dikurangi aktual; positif berarti terlalu tinggi. WAPE: total error absolut dibanding total limbah, bukan akurasi klasifikasi.

Waktu prediksi berikut diukur untuk satu batch tanpa preprocessing dan pemuatan model. Milidetik per baris adalah waktu batch dibagi jumlah baris, bukan benchmark permintaan tunggal. Waktu training mencakup pelatihan dan pemantauan kurva kandidat terpilih, termasuk impor awal library/pengukuran gradien; total tuning mencakup kedua kandidat. Model tidak dilatih ulang pada validasi.

In [8]:
result_rows = []
protocol = hashlib.sha256((ROOT/'src/experiment.py').read_bytes()+(ROOT/'dataset/raw/train.csv').read_bytes()).hexdigest()
for method,item in selected.items():
    d=data[method];label=f'{MODEL}_{method}'
    row={'model':MODEL,'preprocessing':method,'config':json.dumps(item['config']),'protocol_sha256':protocol,
         'train_time_s':item['duration'],'tuning_total_time_s':float(tuning.loc[tuning.preprocessing==method,'train_time_s'].sum()),
         **{f'n_{s}':len(v) for s,v in d['y'].items()}}
    for split in ['train','validation','test']:
        start=time.perf_counter()
        prediction=predict(item['model'],d['features'][split],MODEL,d['ys'])
        elapsed=time.perf_counter()-start
        row.update({f'{split}_{k}':v for k,v in metrics(d['y'][split],prediction).items()})
        row[f'{split}_prediction_time_s']=elapsed
        row[f'{split}_ms_per_row']=elapsed/len(prediction)*1000
        pd.DataFrame({'row':df.loc[idx[split],'_row'],'date':df.loc[idx[split],'date'],'actual':d['y'][split],
                      'prediction':prediction,'residual':d['y'][split]-prediction}).to_csv(OUT/f'{label}_{split}_predictions.csv',index=False)
    save_json(OUT/f'{label}_metrics.json',row)
    result_rows.append(row)
hasil=pd.DataFrame(result_rows)
for split in ['train','validation','test']:
    display(Markdown(f'### Metrik {split}'))
    display(hasil[['model','preprocessing']+[f'{split}_{m}' for m in ['MAE','RMSE','R2','MedAE','Bias','WAPE_percent']]])
display(hasil[['model','preprocessing','train_time_s','tuning_total_time_s','test_prediction_time_s','test_ms_per_row']])


### Metrik train

,model,preprocessing,train_MAE,train_RMSE,train_R2,train_MedAE,train_Bias,train_WAPE_percent
0,RandomForest,standard,5.4403,8.4235,0.8963,4.1523,-0.0011,12.3020
1,RandomForest,robust,2.9870,4.6071,0.9690,2.3047,0.0068,6.7546


### Metrik validation

,model,preprocessing,validation_MAE,validation_RMSE,validation_R2,validation_MedAE,validation_Bias,validation_WAPE_percent
0,RandomForest,standard,9.8086,13.7094,0.8435,7.5323,-0.6138,20.6464
1,RandomForest,robust,9.2478,12.6380,0.8670,7.8892,-0.7663,19.4660


### Metrik test

,model,preprocessing,test_MAE,test_RMSE,test_R2,test_MedAE,test_Bias,test_WAPE_percent
0,RandomForest,standard,8.8672,11.0680,0.8197,7.5352,1.2169,20.1742
1,RandomForest,robust,8.8671,11.0356,0.8208,7.6906,0.9962,20.1740


,model,preprocessing,train_time_s,tuning_total_time_s,test_prediction_time_s,test_ms_per_row
0,RandomForest,standard,0.5859,1.1690,0.0361,0.2073
1,RandomForest,robust,0.7363,1.3173,0.0818,0.4699


## Langkah 9 Melihat prediksi dan kasus yang paling sulit
Garis diagonal menandai prediksi sempurna. Residual positif berarti model meremehkan limbah. Histogram yang lebar berarti variasi error besar. Tabel error terbesar membantu mencari kondisi operasional yang sulit, tetapi tidak digunakan lagi untuk menyetel model; jika dilakukan, dibutuhkan test baru.

In [9]:
for method in ['standard','robust']:
    pred=pd.read_csv(OUT/f'{MODEL}_{method}_test_predictions.csv')
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    axes[0].scatter(pred.actual,pred.prediction,alpha=.6)
    limits=[min(pred.actual.min(),pred.prediction.min()),max(pred.actual.max(),pred.prediction.max())]
    axes[0].plot(limits,limits,'r--');axes[0].set(xlabel='Aktual kg',ylabel='Prediksi kg',title=f'{MODEL} {method}')
    axes[1].scatter(pred.prediction,pred.residual,alpha=.6);axes[1].axhline(0,color='red');axes[1].set(xlabel='Prediksi kg',ylabel='Aktual - prediksi kg')
    axes[2].hist(pred.residual,bins=20);axes[2].axvline(0,color='red');axes[2].set_title('Sebaran error kg')
    fig.tight_layout();fig.savefig(OUT/f'{MODEL}_{method}_diagnostics.png',dpi=140);display(fig);plt.close(fig)
    context=df[['_row','meals_served','kitchen_staff','special_event','staff_experience']]
    difficult=pred.merge(context,left_on='row',right_on='_row',how='left')
    display(difficult.assign(error_absolut=difficult.residual.abs()).nlargest(10,'error_absolut'))


,row,date,actual,prediction,residual,_row,meals_served,kitchen_staff,special_event,staff_experience,error_absolut
126,451,2024-08-05,167.7235,214.8755,-47.1520,451,3170,14,0,Intermediate,47.1520
28,265,2024-04-12,175.5071,205.1000,-29.5929,265,2440,13,0,EXPERT,29.5929
19,380,2024-03-31,168.8688,193.5068,-24.6380,380,4100,18,0,intermediate,24.6380
155,631,2024-09-05,21.5655,45.6300,-24.0645,631,177,5,1,Intermediate,24.0645
18,696,2024-03-30,41.2591,63.8959,-22.6368,696,484,5,0,EXPERT,22.6368
58,131,2024-05-16,21.5092,43.8155,-22.3063,131,328,12,0,EXPERT,22.3063
87,489,2024-06-17,64.1462,42.6077,21.5386,489,439,15,0,Intermediate,21.5386
15,276,2024-03-27,52.8336,31.5127,21.3209,276,197,11,0,Beginner,21.3209
112,64,2024-07-17,53.3452,33.1029,20.2422,64,259,13,0,Intermediate,20.2422
168,642,2024-09-19,185.6943,205.2299,-19.5356,642,3380,16,0,intermediate,19.5356


,row,date,actual,prediction,residual,_row,meals_served,kitchen_staff,special_event,staff_experience,error_absolut
126,451,2024-08-05,167.7235,214.7626,-47.0391,451,3170,14,0,Intermediate,47.0391
168,642,2024-09-19,185.6943,214.4097,-28.7154,642,3380,16,0,intermediate,28.7154
155,631,2024-09-05,21.5655,48.0793,-26.5138,631,177,5,1,Intermediate,26.5138
58,131,2024-05-16,21.5092,44.3193,-22.8102,131,328,12,0,EXPERT,22.8102
125,794,2024-08-04,51.5225,29.9237,21.5988,794,177,9,0,Intermediate,21.5988
15,276,2024-03-27,52.8336,31.3058,21.5278,276,197,11,0,Beginner,21.5278
87,489,2024-06-17,64.1462,42.8725,21.2737,489,439,15,0,Intermediate,21.2737
157,518,2024-09-07,67.2923,46.2884,21.0038,518,485,15,0,Intermediate,21.0038
112,64,2024-07-17,53.3452,32.5170,20.8281,64,259,13,0,Intermediate,20.8281
150,737,2024-08-31,48.2289,27.7864,20.4425,737,226,10,0,intermediate,20.4425


## Langkah 10 Menilai fitur yang membantu prediksi
Permutation importance mengacak satu fitur pada validasi sebanyak tiga kali. Kenaikan MAE positif berarti fitur tersebut membantu model. Nilai negatif dapat terjadi akibat noise atau model yang terlalu mengandalkan fitur tersebut. Ini bukan bukti kausal. Rasio dan porsi saling berkaitan sehingga importance dapat terbagi. Pada LSTM, satu jalur fitur sepanjang jendela diacak sebagai satu blok; hasil bersifat diagnostik.

In [10]:
for method,item in selected.items():
    d=data[method];rng=np.random.default_rng(SEED);importance=[]
    for j,feature in enumerate(d['names']):
        differences=[]
        for repeat in range(3):
            shuffled=d['features']['validation'].copy();order=rng.permutation(len(shuffled))
            if MODEL=='LSTM':shuffled[:,:,j]=shuffled[order,:,j]
            else:shuffled.iloc[:,j]=shuffled.iloc[order,j].to_numpy()
            score=metrics(d['y']['validation'],predict(item['model'],shuffled,MODEL,d['ys']))['MAE']
            differences.append(score-item['score']['MAE'])
        importance.append({'feature':feature,'MAE_increase':np.mean(differences),'std':np.std(differences)})
    imp=pd.DataFrame(importance).sort_values('MAE_increase',ascending=False)
    imp.to_csv(OUT/f'{MODEL}_{method}_importance.csv',index=False)
    display(imp)
    top=imp.head(12).sort_values('MAE_increase')
    fig,ax=plt.subplots(figsize=(9,4));ax.barh(top.feature,top.MAE_increase,xerr=top['std'])
    ax.set(xlabel='Kenaikan MAE validasi kg',title=f'Pengaruh fitur {MODEL} {method}')
    fig.tight_layout();display(fig);plt.close(fig)


,feature,MAE_increase,std
8,staff_per_portion,4.9460,0.0991
0,meals_served,4.0777,0.7124
9,portion_per_staff,0.2357,0.0139
10,special_event_portion,0.2299,0.0695
1,kitchen_staff,0.1706,0.0510
2,temperature_C,0.0596,0.0872
11,staff_experience_Beginner,0.0421,0.0187
4,special_event,0.0405,0.0104
6,month_sin,0.0395,0.0429
3,humidity_percent,0.0391,0.1476


,feature,MAE_increase,std
0,meals_served,12.3637,1.0939
6,month_sin,0.4300,0.2776
10,special_event_portion,0.1787,0.0790
1,kitchen_staff,0.1478,0.0833
11,staff_experience_beginner,0.0625,0.0282
9,portion_per_staff,0.0476,0.0218
7,month_cos,0.0107,0.0419
8,staff_per_portion,-0.0006,0.0345
4,special_event,-0.0055,0.0149
5,is_weekend,-0.0070,0.0068


## Langkah 11 Menyimpan model bersama preprocessing
Model saja tidak cukup. Kategori, median, batas outlier, skala, dan urutan fitur harus sama saat digunakan kembali. Model lama milik tim tidak ditimpa: artefak baru memakai nama model/metode tersendiri di folder models/best_model yang sudah ada. Integrasi ke API merupakan langkah terpisah.

In [11]:
for method,item in selected.items():
    label=f'{MODEL}_{method}'
    if MODEL=='LSTM':item['model'].save(ART/f'{label}.keras')
    else:joblib.dump(item['model'],ART/f'{label}.joblib')
    joblib.dump({'preprocessor':data[method]['prep'],'target_scaler':data[method]['ys'],'window':WINDOW,'name':MODEL},ART/f'{label}_preprocessing.joblib')
    print('Tersimpan:',label)


Tersimpan: RandomForest_standard
Tersimpan: RandomForest_robust


## Langkah 12 Menulis insight dari angka yang benar-benar diperoleh
Kesimpulan berikut berubah mengikuti hasil. MAE test tidak dipakai memilih pemenang. Robust bisa lebih buruk jika clipping menghilangkan nilai ekstrem yang sebenarnya bermakna. Satu split/seed tidak membuktikan keunggulan universal. Setelah notebook model selesai, buka notebook 08 untuk perbandingan delapan eksperimen.

In [12]:
a=hasil.set_index('preprocessing').loc['standard'];b=hasil.set_index('preprocessing').loc['robust']
delta=a.validation_MAE-b.validation_MAE
display(Markdown(f'**Dampak Robust pada validasi:** MAE berubah dari {a.validation_MAE:.3f} menjadi {b.validation_MAE:.3f} kg. Selisih Standard − Robust = {delta:.3f} kg. Positif berarti Robust lebih baik.'))
baseline_value=float(df.loc[idx['train'],TARGET].median())
baseline_mae=metrics(df.loc[idx['test'],TARGET].to_numpy(),np.full(len(idx['test']),baseline_value))['MAE']
for _,r in hasil.iterrows():
    improvement=(baseline_mae-r.test_MAE)/baseline_mae*100
    display(Markdown(f"**{r.preprocessing}:** training MAE {r.train_MAE:.3f} kg, validasi {r.validation_MAE:.3f} kg, test {r.test_MAE:.3f} kg. Selisih validasi − training {r.validation_MAE-r.train_MAE:.3f} kg. R² test {r.test_R2:.3f}. Pengurangan error dibanding tebakan median training pada test {improvement:.1f}%."))
display(Markdown('Gunakan tabel error terbesar dan grafik residual untuk menjelaskan kelemahan model. Jangan mengubah parameter berdasarkan kasus test lalu tetap menyebut hasil test tersebut evaluasi independen.'))


**Dampak Robust pada validasi:** MAE berubah dari 9.809 menjadi 9.248 kg. Selisih Standard − Robust = 0.561 kg. Positif berarti Robust lebih baik.

**standard:** training MAE 5.440 kg, validasi 9.809 kg, test 8.867 kg. Selisih validasi − training 4.368 kg. R² test 0.820. Pengurangan error dibanding tebakan median training pada test 31.4%.

**robust:** training MAE 2.987 kg, validasi 9.248 kg, test 8.867 kg. Selisih validasi − training 6.261 kg. R² test 0.821. Pengurangan error dibanding tebakan median training pada test 31.4%.

Gunakan tabel error terbesar dan grafik residual untuk menjelaskan kelemahan model. Jangan mengubah parameter berdasarkan kasus test lalu tetap menyebut hasil test tersebut evaluasi independen.

## Langkah 13 Membuka kembali artefak untuk melanjutkan pekerjaan
Contoh ini memeriksa model tersimpan pada lima observasi test yang sama, bukan pengujian baru. Jika prediksinya sama, pasangan model/preprocessing dapat dibuka kembali tanpa training. Saat dipakai pada input baru, urutan kolom dan asumsi ketersediaan fitur harus tetap sama. Model LSTM membutuhkan konteks, bukan hanya satu baris.

In [13]:
for method in ['standard','robust']:
    label=f'{MODEL}_{method}'
    bundle=joblib.load(ART/f'{label}_preprocessing.joblib')
    if MODEL=='LSTM':
        import tensorflow as tf
        loaded=tf.keras.models.load_model(ART/f'{label}.keras',compile=False)
        encoded=bundle['preprocessor'].transform(df,scaled=True)
        sample=sequence(encoded,idx['test'][:5])
    else:
        loaded=joblib.load(ART/f'{label}.joblib')
        transformed=bundle['preprocessor'].native(df) if MODEL=='CatBoost' else bundle['preprocessor'].transform(df)
        sample=transformed.iloc[idx['test'][:5]]
    restored=predict(loaded,sample,MODEL,bundle['target_scaler'])
    saved=pd.read_csv(OUT/f'{label}_test_predictions.csv').prediction.iloc[:5].to_numpy()
    np.testing.assert_allclose(restored,saved,rtol=1e-5,atol=1e-5)
    display(pd.DataFrame({'prediksi_tersimpan':saved,'prediksi_setelah_dibuka':restored}))
    print(method,': LULUS, model dapat dibuka kembali')


,prediksi_tersimpan,prediksi_setelah_dibuka
0,40.9711,40.9711
1,43.0028,43.0028
2,32.9143,32.9143
3,38.6253,38.6253
4,42.3436,42.3436


standard : LULUS, model dapat dibuka kembali


,prediksi_tersimpan,prediksi_setelah_dibuka
0,42.9067,42.9067
1,43.8147,43.8147
2,29.4538,29.4538
3,39.0333,39.0333
4,41.8757,41.8757


robust : LULUS, model dapat dibuka kembali
